In [ ]:
#Application Name: Telemetry Analysis Agent
#Framework: LlamaIndex
#LLM: OpenAI GPT-4o
#Purpose: Analyze application telemetry, retrieve similar historical incidents, #identify probable root cause, and recommend corrective actions.

In [1]:
# Install the main LlamaIndex packages required for the demo

!pip install -q llama-index llama-index-llms-openai llama-index-embeddings-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 17.8 MB/s eta 0:00:00


In [2]:
# Import required Python and LlamaIndex libraries

import os

from llama_index.core import Document, VectorStoreIndex
from llama_index.core.tools import QueryEngineTool
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

In [3]:
# Get the OpenAI API key
# In Google Colab, store OPENAI_API_KEY in Secrets

from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OpenAI API key configured.")

OpenAI API key configured.


In [4]:
# LLM is used for telemetry analysis and generating the final answer
# Embedding model converts historical incidents into vectors

llm = OpenAI(
    model="gpt-4o",
    temperature=0
)

embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

print("LLM and embedding model configured.")

LLM and embedding model configured.


In [5]:
# Sample historical incidents
# In a real application, this data could come from:
# Azure Monitor, Application Insights, Splunk, Datadog, CloudWatch, etc.

historical_incidents = [

    """
    Incident: Claims API high latency
    CPU: 92%
    Memory: 78%
    Error Rate: 2%
    Response Time: 4800 ms
    Root Cause: Database connection pool exhaustion.
    Resolution: Increased connection pool and optimized long-running queries.
    """,

    """
    Incident: Claims API high latency
    CPU: 88%
    Memory: 75%
    Error Rate: 1%
    Response Time: 4200 ms
    Root Cause: Slow SQL queries after a database deployment.
    Resolution: Added indexes and optimized stored procedures.
    """,

    """
    Incident: Claims API errors
    CPU: 45%
    Memory: 82%
    Error Rate: 18%
    Response Time: 1200 ms
    Root Cause: Expired authentication token configuration.
    Resolution: Updated identity provider configuration and restarted services.
    """,

    """
    Incident: Claims API unavailable
    CPU: 95%
    Memory: 91%
    Error Rate: 35%
    Response Time: 9000 ms
    Root Cause: Memory leak in application service.
    Resolution: Restarted affected instances and deployed memory leak fix.
    """
]

print(f"Loaded {len(historical_incidents)} historical incidents.")

Loaded 4 historical incidents.


In [6]:
# Convert each historical incident into a LlamaIndex Document

documents = [
    Document(text=incident)
    for incident in historical_incidents
]

print(f"Created {len(documents)} LlamaIndex documents.")

Created 4 LlamaIndex documents.


In [7]:
# Create a vector index from historical telemetry incidents
# LlamaIndex automatically creates embeddings and stores them in memory

index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model
)

print("Telemetry vector index created.")

Telemetry vector index created.


In [8]:
# Create a query engine to retrieve similar historical incidents

query_engine = index.as_query_engine(
    similarity_top_k=3
)

# Convert the query engine into a tool that the agent can use

telemetry_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="historical_telemetry_search",
    description=(
        "Search historical telemetry incidents to find similar problems, "
        "root causes, and previous resolutions."
    )
)

print("Telemetry search tool created.")

Telemetry search tool created.


In [9]:
# Import LlamaIndex FunctionAgent

from llama_index.core.agent.workflow import FunctionAgent

# Create the agent
# The agent decides when to use the historical telemetry tool

telemetry_agent = FunctionAgent(
    tools=[telemetry_tool],
    llm=llm,
    system_prompt="""
    You are a Telemetry Analysis Agent.

    Your job is to analyze current application telemetry.

    When telemetry indicates a problem:
    1. Search historical telemetry for similar incidents.
    2. Compare the current metrics with historical incidents.
    3. Identify the most likely root cause.
    4. Recommend a practical resolution.
    5. Clearly explain the reasoning.

    Do not invent telemetry values.
    If the available evidence is insufficient, say so.
    """
)

print("Telemetry Agent created.")

Telemetry Agent created.


In [10]:
# Simulate current telemetry received from a monitoring system

current_telemetry = """
Current Telemetry:

Service: Claims API

CPU Usage: 90%
Memory Usage: 79%
Error Rate: 2%
Average Response Time: 4500 ms

Recent deployment: No

Question:
Why is the Claims API experiencing high latency?
What is the likely root cause and recommended action?
"""

print(current_telemetry)


Current Telemetry:

Service: Claims API

CPU Usage: 90%
Memory Usage: 79%
Error Rate: 2%
Average Response Time: 4500 ms

Recent deployment: No

Question:
Why is the Claims API experiencing high latency?
What is the likely root cause and recommended action?



In [11]:
# Send the current telemetry to the LlamaIndex agent

response = await telemetry_agent.run(
    current_telemetry
)

print(response)

The current high latency in the Claims API is likely due to inefficient database queries. Historical telemetry indicates that similar incidents were caused by slow SQL queries, which were resolved by adding indexes and optimizing stored procedures.

### Likely Root Cause:
- **Inefficient Database Queries**: The high CPU usage (90%) and elevated average response time (4500 ms) suggest that the API is spending excessive time processing requests, likely due to slow database operations.

### Recommended Action:
- **Optimize Database Queries**: Review and optimize the SQL queries used by the Claims API. This may involve adding indexes to frequently queried columns and optimizing stored procedures to improve execution efficiency.

### Reasoning:
The combination of high CPU usage and increased response time, without a recent deployment, points towards a backend processing issue rather than a code change. Historical data supports this by showing that similar latency issues were resolved throug

In [12]:
# Ask telemetry-related questions interactively

while True:

    question = input("\nEnter telemetry question (or 'exit'): ")

    if question.lower() == "exit":
        print("Telemetry Agent stopped.")
        break

    response = await telemetry_agent.run(question)

    print("\nAgent Response:")
    print(response)


Enter telemetry question (or 'exit'): Why is the Claims API experiencing high latency? Compare the current telemetry with similar historical incidents and identify the most likely root cause and recommended action.

Agent Response:
The current high latency issue with the Claims API is similar to a historical incident where the latency was caused by slow SQL queries following a database deployment. In that case, the root cause was identified as inefficient database operations, and the resolution involved adding indexes and optimizing stored procedures.

### Recommended Action:
1. **Database Query Analysis**: Analyze the current SQL queries being executed by the Claims API to identify any that are performing poorly.
2. **Index Optimization**: Check if there are missing indexes that could improve query performance.
3. **Stored Procedure Review**: Review and optimize stored procedures to ensure they are efficient.
4. **Monitor Database Performance**: Use database monitoring tools to track